# Amazon Nova RFT Single-Turn Training - Quick Start Guide

This notebook provides a comprehensive walkthrough of Reinforcement Fine-Tuning (RFT) for Nova models.

## Table of Contents

1. [What You'll Learn](#what-youll-learn)
2. [Prerequisites](#prerequisites)
3. [What is RFT?](#what-is-rft)
4. [Step 1: Import Required Modules](#step-1-import-required-modules)
5. [Step 2: Configure Your AWS Resources](#step-2-configure-your-aws-resources)
6. [Step 3: Create Training Reward Function](#step-3-create-training-reward-function)
7. [Step 4: Prepare RFT Training Dataset](#step-4-prepare-rft-training-dataset)
8. [Step 5: Deploy Reward Function to Lambda](#step-5-deploy-reward-function-to-lambda)
9. [Step 6: Configure Runtime and Start Training](#step-6-configure-runtime-and-start-training)
10. [Step 7: Monitor Training Progress](#step-7-monitor-training-progress)
11. [Step 8: Create Evaluation Reward Function and Run Evaluation](#step-8-create-evaluation-reward-function-and-run-evaluation)
12. [Step 9: Deploy Your Model](#step-9-deploy-your-model)
13. [Troubleshooting Lambda Issues](#troubleshooting-lambda-issues)
14. [Summary](#summary)

## What You'll Learn

1. Creating reward functions for RFT training
2. Preparing RFT datasets with reference answers
3. Training Nova models with RFT (automatic Lambda verification)
4. Creating evaluation reward functions with metrics
5. Evaluating RFT-trained models
6. Troubleshooting common Lambda issues
7. Platform considerations (SMHP vs SMTJ)

## Prerequisites

- AWS credentials configured
- S3 bucket for data and model artifacts
- IAM permissions for SageMaker, Lambda, and Bedrock
- Nova Forge SDK installed

## What is RFT?

Reinforcement Fine-Tuning (RFT) uses reward functions to guide model training:
- Uses a reward function to score model outputs
- Optimizes the model to maximize reward scores
- Allows custom evaluation metrics
- Supports both training and evaluation modes
- SDK automatically verifies Lambda functions before training

## Step 1: Import Required Modules

In [ ]:
!cd ../ && pip install .

In [ ]:
import json
import os

import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound


def load_credentials(profile=None):
    if profile:
        session = boto3.Session(profile_name=profile)
    else:
        session = boto3.Session()

    credentials = session.get_credentials()
    if not credentials:
        raise RuntimeError("No credentials found")

    return {
        "aws_access_key_id": credentials.access_key,
        "aws_secret_access_key": credentials.secret_key,
        "aws_session_token": credentials.token,
        "region_name": session.region_name or "us-east-1",
    }

In [ ]:
creds = load_credentials()

In [ ]:
from amzn_nova_forge_sdk import *

print("✅ SDK imported successfully!")

## Step 2: Configure Your AWS Resources

In [ ]:
# TODO: Update these values
S3_BUCKET = "your-bucket-name"  # TODO: Replace
S3_PREFIX = "rft-singleturn-demo"
S3_DATA_PATH = f"s3://{S3_BUCKET}/{S3_PREFIX}/input"
S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/{S3_PREFIX}/output"

print(f"Data Path: {S3_DATA_PATH}")
print(f"Output Path: {S3_OUTPUT_PATH}")

## Step 3: Create Training Reward Function

The training reward function must return:
- `id`: Sample identifier
- `aggregate_reward_score`: Numeric reward score

In [ ]:
# You can also use the existing samples/my_reward.py file
# For this demo, we'll create a simple reward function inline

training_reward_code = """
import json
from dataclasses import asdict, dataclass
from typing import List

@dataclass
class RewardOutput:
    id: str
    aggregate_reward_score: float

def lambda_handler(event, context):
    scores = []
    
    for sample in event:
        sample_id = sample.get("id", "unknown")
        messages = sample.get("messages", [])
        reference = sample.get("reference_answer", "")
        
        # Extract model response
        completion = ""
        for msg in reversed(messages):
            if msg.get("role") in ["assistant", "nova_assistant"]:
                completion = msg.get("content", "")
                break
        
        # Calculate reward
        if completion.strip().lower() == reference.strip().lower():
            reward = 1.0  # Exact match
        elif reference.lower() in completion.lower():
            reward = 0.5  # Partial match
        else:
            reward = -1.0  # No match
        
        scores.append(RewardOutput(id=sample_id, aggregate_reward_score=reward))
    
    return [asdict(score) for score in scores]
"""

with open("rft_training_reward.py", "w") as f:
    f.write(training_reward_code)

print("✅ Training reward function created")

## Step 4: Prepare RFT Training Dataset

RFT datasets must include:
- `id`: Unique identifier
- `messages`: Conversation in Converse format
- `reference_answer`: Ground truth (optional but recommended)

In [ ]:
# Create RFT training data
rft_training_data = [
    {
        "id": f"sample_{i}",
        "messages": [
            {"role": "user", "content": "What is machine learning?"},
            {
                "role": "assistant",
                "content": "Machine learning is a subset of AI that enables systems to learn from data.",
            },
        ],
        "reference_answer": "Machine learning is a subset of artificial intelligence.",
    }
    for i in range(100)
]

# Add more diverse samples
rft_training_data.extend(
    [
        {
            "id": f"sample_{100 + i}",
            "messages": [
                {"role": "user", "content": "Explain AWS in one sentence."},
                {"role": "assistant", "content": "AWS is a cloud computing platform."},
            ],
            "reference_answer": "AWS is Amazon's cloud computing platform.",
        }
        for i in range(100)
    ]
)

# Save locally
with open("rft_training_data.jsonl", "w") as f:
    for item in rft_training_data:
        f.write(json.dumps(item) + "\n")

print(f"✅ Created {len(rft_training_data)} training samples")

In [ ]:
# Load and upload to S3
loader = JSONLDatasetLoader()
loader.load("rft_training_data.jsonl")

# Preview
print("\n Dataset Preview:")
loader.show(n=2)

# Upload to S3
rft_train_s3_path = loader.save_data(f"{S3_DATA_PATH}/rft_train.jsonl")
print(f"\n✅ Training data uploaded to: {rft_train_s3_path}")

## Step 5: Deploy Reward Function to Lambda

**Important**: You must deploy your reward function to AWS Lambda before training.

### Lambda Requirements:
- **SMHP Platform**: Function name must contain 'SageMaker' (case-insensitive)
- **SMTJ Platform**: No naming restrictions

In [ ]:
# After deploying to Lambda, set your Lambda ARN here
# TODO: Replace with your actual Lambda ARN
LAMBDA_ARN = "arn:aws:lambda:us-east-1:123456789012:function:MySageMakerRFTReward"

print(f"Lambda ARN: {LAMBDA_ARN}")
print(" Make sure to replace with your actual Lambda ARN before training!")

## Step 6: Configure Runtime and Start Training

The SDK automatically verifies your Lambda function before training starts when you enable `rft_lambda` in `validation_config`.

### Automatic Lambda Verification

When `validation_config={'rft_lambda': True}` is set:

1. **Before Training**: SDK reads sample data from your `data_s3_path`
2. **Lambda Test**: Invokes your Lambda with test samples
3. **Format Check**: Validates response format (checks for required fields)
4. **Strict Validation**: Raises error if ANY samples fail
5. **Training Start**: Only proceeds if all samples pass

This catches configuration errors early and prevents expensive training runs with broken Lambda functions.

#### Configuration Options

#### Simple: Enable with default settings (10 samples)
`validation_config={'rft_lambda': True}`

#### Advanced: Specify number of samples to test
`validation_config={'rft_lambda': {'enabled': True, 'samples': 20}}`

#### Disable: Skip Lambda verification (not recommended)
`validation_config={'rft_lambda': False}`


In [ ]:
# Configure runtime (SMTJ example)
runtime = SMTJRuntimeManager(
    instance_type="ml.p5.48xlarge",
    instance_count=4,
)

platform = Platform.SMTJ

print("✅ Runtime configured")
print(f"   Instance Type: {runtime.instance_type}")
print(f"   Instance Count: {runtime.instance_count}")

In [ ]:
# For SMHP, use this instead:
# runtime = SMHPRuntimeManager(
#     instance_type="ml.p5.48xlarge",
#     instance_count=4,
#     cluster_name="your-cluster-name",
#     namespace="your-namespace",
# )
# platform = Platform.SMHP

In [ ]:
# Create MLflow monitor
mlflow_monitor = MLflowMonitor(tracking_uri="<MLflow Server/App Uri>")

# Create RFT customizer with Lambda verification enabled
rft_customizer = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,  # RFT supported for Nova Lite 2.0
    method=TrainingMethod.RFT_LORA,
    infra=runtime,
    data_s3_path=rft_train_s3_path,
    output_s3_path=f"{S3_OUTPUT_PATH}/rft-training",
    validation_config={
        "iam": True,
        "infra": True,
        "rft_lambda": True,  # Enable automatic Lambda verification before training
    },
    mlflow_monitor=mlflow_monitor,
)

print("✅ RFT Customizer initialized")
print(f"   Model: Nova Lite 2.0")
print(f"   Method: RFT with LoRA")
print(f"   Lambda Verification: Enabled")

In [ ]:
# Start RFT training
training_config = {
    "max_steps": 100,
    "save_steps": 30,
    "lr": 5e-6,
    "global_batch_size": 64,
}

# TODO: Uncomment after setting Lambda ARN
training_result = rft_customizer.train(
    job_name="rft-singleturn-training",
    overrides=training_config,
    rft_lambda_arn=LAMBDA_ARN,
)

print("\n🚀 RFT Training started!")
print(f"   Job ID: {training_result.job_id}")
print(f"   Checkpoint: {training_result.model_artifacts.checkpoint_s3_path}")
print(f"   Output: {training_result.model_artifacts.output_s3_path}")

job_id = training_result.job_id

In [ ]:
# Save training result for later use
training_result.dump("training_result.json")
print("✅ Training result saved to training_result.json")

In [ ]:
# Load training result from file
loaded_training_result = TrainingResult.load("training_result.json")
print("✅ Training result loaded from file")
print(f"   Job ID: {loaded_training_result.job_id}")
print(f"   Checkpoint: {loaded_training_result.model_artifacts.checkpoint_s3_path}")

## Step 7: Monitor Training Progress

In [ ]:
# View training logs (while training is running)
rft_customizer.get_logs(limit=50, start_from_head=False)

In [ ]:
# After training completes, view full logs
monitor = CloudWatchLogMonitor.from_job_id(job_id=job_id, platform=platform)
monitor.show_logs(limit=100, start_from_head=True)

## Step 8: Create Evaluation Reward Function and Run Evaluation

Evaluation reward functions can return additional metrics beyond the aggregate score.

In [ ]:
# Create evaluation dataset
eval_data = [
    {
        "id": f"eval_{i}",
        "messages": [
            {"role": "user", "content": "What is cloud computing?"},
            {
                "role": "assistant",
                "content": "Cloud computing delivers computing services over the internet.",
            },
        ],
        "reference_answer": "Cloud computing provides on-demand computing resources.",
    }
    for i in range(50)
]

with open("rft_eval_data.jsonl", "w") as f:
    for item in eval_data:
        f.write(json.dumps(item) + "\n")

print(f"✅ Created {len(eval_data)} evaluation samples")

In [ ]:
# Upload evaluation data
eval_loader = JSONLDatasetLoader()
eval_loader.load("rft_eval_data.jsonl")
eval_s3_path = eval_loader.save_data(f"{S3_DATA_PATH}/rft_eval.jsonl")

print(f"✅ Evaluation data uploaded to: {eval_s3_path}")

In [ ]:
# Configure evaluation runtime
eval_runtime = SMTJRuntimeManager(
    instance_type="ml.p5.48xlarge",
    instance_count=1,
)

# Create evaluator with Lambda verification
evaluator = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.EVALUATION,
    infra=eval_runtime,
    data_s3_path=eval_s3_path,
    output_s3_path=f"{S3_OUTPUT_PATH}/rft-evaluation",
    validation_config={
        "iam": True,
        "infra": True,
        "rft_lambda": True,  # Enable automatic Lambda verification
    },
)

print("✅ Evaluator initialized")
print(f"   Lambda Verification: Enabled")

In [ ]:
# TODO: Deploy eval reward function to Lambda and set ARN
EVAL_LAMBDA_ARN = "arn:aws:lambda:us-east-1:123456789012:function:MySageMakerRFTEval"

# Run evaluation (uncomment after deploying eval Lambda)
eval_result = evaluator.evaluate(
    job_name="rft-singleturn-eval",
    eval_task=EvaluationTask.GEN_QA,
    data_s3_path=eval_s3_path,
    # model_path=training_result.model_artifacts.checkpoint_s3_path,  # Use trained model
    processor={"lambda_arn": EVAL_LAMBDA_ARN},
)

print("\n🚀 Evaluation started!")
print(f"   Job ID: {eval_result.job_id}")
print(f"   Output: {eval_result.eval_output_path}")

In [ ]:
# Save evaluation result for later use
eval_result.dump("eval_result.json")
print("✅ Evaluation result saved to eval_result.json")

In [ ]:
# Load evaluation result from file
loaded_eval_result = EvaluationResult.load("eval_result.json")
print("✅ Evaluation result loaded from file")
print(f"   Job ID: {loaded_eval_result.job_id}")
print(f"   Output: {loaded_eval_result.eval_output_path}")

## Step 9: Deploy Your Model

Deploy your RFT-trained model to SageMaker or Bedrock.

In [ ]:
# Deploy to SageMaker (uncomment after training completes)
# deployment_result = rft_customizer.deploy(
#     deploy_platform=DeployPlatform.SAGEMAKER,
#     unit_count=1,
#     endpoint_name="rft-singleturn-model",
#     sagemaker_environment_variables={
#         "CONTEXT_LENGTH": "12000",
#         "MAX_CONCURRENCY": "16",
#     },
#     job_result=training_result,
# )
#
# print("\n🚀 Model deployment started!")
# print(f"   Endpoint: {deployment_result.endpoint.endpoint_name}")
# print(f"   Status: {deployment_result.status}")

In [ ]:
# Test inference (after deployment)
# inference_result = rft_customizer.invoke_inference(
#     request_body={
#         "messages": [{"role": "user", "content": "What is machine learning?"}],
#         "max_tokens": 100,
#         "stream": False,
#     }
# )
#
# inference_result.show()

## Summary

You've completed the RFT single-turn training workflow:

✅ Created and verified training reward function  
✅ Prepared RFT dataset with reference answers  
✅ Configured runtime infrastructure  
✅ Started RFT training with Lambda reward function  
✅ Created evaluation reward function with metrics  
✅ Verified evaluation reward function  
✅ Ran RFT evaluation  
✅ Deployed the model

## Next Steps

- Experiment with different reward calculation strategies
- Add more sophisticated metrics (BLEU, ROUGE, etc.)
- Try multi-turn RFT training (see `docs/rft_multiturn.md`)
- Fine-tune hyperparameters for better performance
- Deploy to production with appropriate monitoring

## Resources

- [AWS RFT Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/nova-hp-rft-nova2.html)
- SDK Spec: `docs/spec.md`
- RFT Multi-turn: `docs/rft_multiturn.md`